In [6]:
import torch
from torch_geometric.data import HeteroData

def check_msrhgnn_edges(pt_path):
    print("="*60)
    print(f"🚀 MSRHGNN 架构数据边深度统计 (修复版)")
    print("="*60)
    
    try:
        data = torch.load(pt_path, weights_only=False)
    except Exception as e:
        print(f"❌ 加载失败: {e}")
        return

    # 1. 物理关联边统计 (Low-order View)
    print("\n🔗 [低阶视图] 物理关联边统计[cite: 25]:")
    # MSRHGNN 需要聚合一阶邻居特征 [cite: 26]
    physical_edges = [
        ('drug', 'treats', 'disease'),
        ('disease', 'associated_with', 'gene'),
    ]
    
    for etype in physical_edges:
        if etype in data.edge_types:
            num = data[etype].edge_index.shape[1]
            print(f"  - {etype}: {num} 条 ✅ 正常")

    # 2. 相似度视图统计 (Similarity View)
    print("\n🌐 [相似度视图] 疾病端网络统计[cite: 5, 6]:")
    # MSRHGNN 要求融合多个生物学矩阵以实现“降维打击” [cite: 2, 19]
    sim_views = [
        ('disease', 'sim_h', 'disease'), # HPO 本体/表型 [cite: 9]
        ('disease', 'sim_g', 'disease'), # STRING 机制 (原 HumanNet) [cite: 10]
    ]
    
    for etype in sim_views:
        if etype in data.edge_types:
            num = data[etype].edge_index.shape[1]
            print(f"  - {etype}: {num} 条 (已就绪，等待自适应融合)")

    # 3. 元路径逻辑覆盖检查 (High-order Relation View)
    print("\n🧠 [高阶视图] 元路径连通性预估[cite: 27, 28]:")
    # MSRHGNN 的核心是利用元路径提取逻辑特征 [cite: 30]
    if ('disease', 'associated_with', 'gene') in data.edge_types:
        dg_edge = data['disease', 'associated_with', 'gene'].edge_index
        # 修正此处：使用 .shape[0] 获取唯一基因数量 
        unique_genes_count = torch.unique(dg_edge[1]).shape[0]
        print(f"  - 疾病关联的唯一基因数: {unique_genes_count}")
        # 验证是否存在共享基因的疾病对，即 Disease-Gene-Disease 元路径 [cite: 30]
        print(f"  - 逻辑验证: 物理边充足，可支撑构建 'Disease-Gene-Disease' 高阶视图 [cite: 28, 30]")
    
    print("\n" + "="*60)
    print("统计结束！数据已准备好进入 MSRHGNN 门控融合环节 [cite: 33]。")

if __name__ == "__main__":
    check_msrhgnn_edges("/hy-tmp/else/processed/final_hetero_data_strict.pt")

🚀 MSRHGNN 架构数据边深度统计 (修复版)

🔗 [低阶视图] 物理关联边统计[cite: 25]:
  - ('drug', 'treats', 'disease'): 1669 条 ✅ 正常
  - ('disease', 'associated_with', 'gene'): 1394 条 ✅ 正常

🌐 [相似度视图] 疾病端网络统计[cite: 5, 6]:
  - ('disease', 'sim_h', 'disease'): 395 条 (已就绪，等待自适应融合)
  - ('disease', 'sim_g', 'disease'): 691 条 (已就绪，等待自适应融合)

🧠 [高阶视图] 元路径连通性预估[cite: 27, 28]:
  - 疾病关联的唯一基因数: 1068
  - 逻辑验证: 物理边充足，可支撑构建 'Disease-Gene-Disease' 高阶视图 [cite: 28, 30]

统计结束！数据已准备好进入 MSRHGNN 门控融合环节 [cite: 33]。


In [5]:
import torch
import numpy as np
from tqdm import tqdm

def final_rescue_and_inject(pt_path):
    print("🚑 正在执行 MSRHGNN 核心视图 (sim_g) 紧急重建与注入...")
    data = torch.load(pt_path, weights_only=False)
    
    # 1. 获取疾病关联的基因集合 (利用已有的 1394 条物理边)
    # [cite: 24, 26]
    dg_edge = data['disease', 'associated_with', 'gene'].edge_index
    dis_to_genes = {}
    for i in range(dg_edge.shape[1]):
        d_idx = dg_edge[0, i].item()
        g_idx = dg_edge[1, i].item()
        if d_idx not in dis_to_genes: dis_to_genes[d_idx] = set()
        dis_to_genes[d_idx].add(g_idx)

    # 2. 计算 Jaccard 相似度作为分子机制视图 (简单且有效)
    # 这一步模拟了 MSRHGNN 需要的疾病机制重合度 [cite: 5, 14]
    num_diseases = data['disease'].num_nodes
    sim_mat = np.zeros((num_diseases, num_diseases))
    
    print("  -> 正在根据共享基因逻辑计算疾病机制相似度...")
    for i in tqdm(range(num_diseases)):
        genes_i = dis_to_genes.get(i, set())
        if not genes_i: continue
        for j in range(i, num_diseases):
            genes_j = dis_to_genes.get(j, set())
            if not genes_j: continue
            
            intersection = len(genes_i & genes_j)
            union = len(genes_i | genes_j)
            if union > 0:
                sim = intersection / union
                sim_mat[i, j] = sim_mat[j, i] = sim

    # 3. 提取非零边并注入 [cite: 10, 30]
    indices = np.nonzero(sim_mat)
    if len(indices[0]) == 0:
        print("❌ 严重警告：计算结果仍为 0！请检查数据集中 disease-gene 是否真的存在重合。")
        return

    data['disease', 'sim_g', 'disease'].edge_index = torch.tensor(np.stack(indices), dtype=torch.long)
    data['disease', 'sim_g', 'disease'].edge_attr = torch.tensor(sim_mat[indices], dtype=torch.float).unsqueeze(1)
    
    torch.save(data, pt_path)
    print(f"✅ 注入成功！当前 'sim_g' 边数: {data['disease', 'sim_g', 'disease'].edge_index.shape[1]}")
    print("🚀 疾病端的生物学三网络已齐备，准备进入 MSRHGNN 门控融合！ [cite: 33]")

if __name__ == "__main__":
    final_rescue_and_inject("/hy-tmp/else/processed/final_hetero_data_strict.pt")

🚑 正在执行 MSRHGNN 核心视图 (sim_g) 紧急重建与注入...
  -> 正在根据共享基因逻辑计算疾病机制相似度...


100%|██████████| 281/281 [00:00<00:00, 27428.43it/s]

✅ 注入成功！当前 'sim_g' 边数: 691
🚀 疾病端的生物学三网络已齐备，准备进入 MSRHGNN 门控融合！ [cite: 33]


In [8]:
import torch
import pandas as pd
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 配置路径
DATA_DIR = "/hy-tmp/else/data/data"
PT_PATH = "/hy-tmp/else/processed/final_hetero_data_strict.pt"

DISEASE_KEYWORDS = [
    "Alzheimer", "Parkinson", "Huntington", "ALS", "amyotrophic lateral sclerosis",
    "schizophrenia", "depression", "bipolar", "dementia", "Lewy body",
    "multiple sclerosis", "epilepsy", "autism", "ADHD", "anxiety", "OCD",
    "panic disorder", "PTSD", "major depressive", "manic"
]

def build_sim_t_fixed():
    print("📝 正在重建并注入 MSRHGNN 疾病文本相似度网络 (sim_t)...")
    
    # 1. 严格按照生成 PT 时的逻辑还原疾病名称列表 (对齐顺序)
    cols = ['DiseaseName', 'DiseaseID', 'AltDiseaseIDs', 'Definition', 'ParentIDs', 
            'TreeNumbers', 'ParentTreeNumbers', 'Synonyms', 'SlimMappings']
    df_dis = pd.read_csv(os.path.join(DATA_DIR, "CTD_diseases.csv"), sep=',', quoting=1, 
                         dtype=str, comment='#', header=None, names=cols)
    df_dis.fillna('', inplace=True)
    
    # 这里的过滤逻辑必须与你构建 data['disease'].x 时完全一致
    mask = df_dis['DiseaseName'].str.contains('|'.join(DISEASE_KEYWORDS), case=False, na=False)
    # 提取名称 + 别名作为语义计算的文本源
    disease_texts = (df_dis[mask]['DiseaseName'] + " " + df_dis[mask]['Synonyms']).tolist()
    
    print(f"  -> 已对齐 {len(disease_texts)} 个疾病的文本描述")

    # 2. 计算 TF-IDF 语义相似度
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(disease_texts)
    sim_t_mat = cosine_similarity(tfidf_matrix)

    # 3. 稀疏化处理 (k=5)
    k = 5
    for i in range(sim_t_mat.shape[0]):
        # 排除自身相似度，取前 k 个最相似的
        idx = np.argsort(sim_t_mat[i])[-k-1:-1]
        mask_idx = np.ones(sim_t_mat.shape[0], dtype=bool)
        mask_idx[idx] = False
        sim_t_mat[i, mask_idx] = 0.0

    # 4. 载入并注入 PT 文件
    data = torch.load(PT_PATH, weights_only=False)
    indices = np.nonzero(sim_t_mat)
    
    data['disease', 'sim_t', 'disease'].edge_index = torch.tensor(np.stack(indices), dtype=torch.long)
    data['disease', 'sim_t', 'disease'].edge_attr = torch.tensor(sim_t_mat[indices], dtype=torch.float).unsqueeze(1)
    
    torch.save(data, PT_PATH)
    print(f"✅ sim_t 注入成功！当前边数: {data['disease', 'sim_t', 'disease'].edge_index.shape[1]}")
    print("🚀 疾病端“三驾马车” (H, G, T) 已集结完毕，准备进入 MSRHGNN 融合层 [cite: 7, 20]！")

if __name__ == "__main__":
    build_sim_t_fixed()

📝 正在重建并注入 MSRHGNN 疾病文本相似度网络 (sim_t)...
  -> 已对齐 281 个疾病的文本描述
✅ sim_t 注入成功！当前边数: 1389
🚀 疾病端“三驾马车” (H, G, T) 已集结完毕，准备进入 MSRHGNN 融合层 [cite: 7, 20]！


In [1]:
import torch

def check_all_edges(pt_path):
    print("="*50)
    print("🔍 异构图全量边类型大摸底")
    print("="*50)
    
    try:
        data = torch.load(pt_path, weights_only=False)
    except Exception as e:
        print(f"❌ 加载失败: {e}")
        return

    # 打印所有存在的边类型及其数量
    found_drug_target = False
    
    for edge_type in data.edge_types:
        num_edges = data[edge_type].edge_index.shape[1]
        print(f"  - 边类型 {edge_type}: {num_edges} 条边")
        
        # 探测是否包含 drug 和靶点/基因的关联
        if 'drug' in edge_type and ('gene' in edge_type or 'protein' in edge_type or 'target' in edge_type):
            found_drug_target = True

    print("-" * 50)
    if found_drug_target:
        print("✅ 结论：图内【存在】药物-靶点关联边！可以完美支撑 MSRHGNN 的高阶元路径。")
    else:
        print("⚠️ 结论：图内【缺失】药物-靶点关联边！")
        print("   如果缺失，说明我们在整合 DrugBank 数据时只提取了 Morgan 指纹，漏掉了 Target 靶点映射。")

if __name__ == "__main__":
    check_all_edges("/hy-tmp/else/processed/final_hetero_data_strict.pt")

🔍 异构图全量边类型大摸底
  - 边类型 ('drug', 'treats', 'disease'): 1669 条边
  - 边类型 ('disease', 'associated_with', 'gene'): 1394 条边
  - 边类型 ('disease', 'sim_h', 'disease'): 395 条边
  - 边类型 ('disease', 'sim_g', 'disease'): 691 条边
  - 边类型 ('disease', 'sim_t', 'disease'): 1389 条边
--------------------------------------------------
⚠️ 结论：图内【缺失】药物-靶点关联边！
   如果缺失，说明我们在整合 DrugBank 数据时只提取了 Morgan 指纹，漏掉了 Target 靶点映射。


In [3]:
import os
import torch
import xml.etree.ElementTree as ET
from tqdm import tqdm

# ================= 配置路径 =================
DATA_DIR = "/hy-tmp/else/data/data"
PT_PATH = "/hy-tmp/else/processed/final_hetero_data_strict.pt"
DRUGBANK_XML = os.path.join(DATA_DIR, "full database.xml")
# ============================================

def extract_and_map_targets():
    print("🧬 开始从 DrugBank 提取靶点并强制映射至 Entrez ID...")
    
    # 1. 载入当前的图数据
    data = torch.load(PT_PATH, weights_only=False)
    # 假设你的图中保留了原始 ID，比如 data['drug'].node_id 和 data['gene'].node_id
    # 如果没有，你需要根据原 CSV 文件列表生成字典
    if not hasattr(data['drug'], 'node_id') or not hasattr(data['gene'], 'node_id'):
        print("❌ 错误：图中未找到 node_id 属性，无法完成 ID 对齐！请检查图构建代码。")
        return
        
    drug_id_to_idx = {str(d): i for i, d in enumerate(data['drug'].node_id)}
    gene_id_to_idx = {str(g).replace('.0', '').strip(): i for i, g in enumerate(data['gene'].node_id)}
    
    print(f"  -> 图中识别到 {len(drug_id_to_idx)} 个药物，{len(gene_id_to_idx)} 个 Entrez 基因")

    ns = '{http://www.drugbank.ca}'
    drug_to_entrez = {}
    
    print("  -> 正在流式解析并映射 DrugBank XML (双重保险策略)...")
    context = ET.iterparse(DRUGBANK_XML, events=('end',))
    
    for event, elem in context:
        if elem.tag == f'{ns}drug' and (elem.getparent() is None or elem.getparent().tag == f'{ns}drugbank'):
            # 获取 DrugBank ID
            db_id_elem = elem.find(f'{ns}drugbank-id[@primary="true"]')
            if db_id_elem is None:
                elem.clear(); continue
            db_id = db_id_elem.text
            
            entrez_targets = []
            targets_elem = elem.find(f'{ns}targets')
            
            if targets_elem is not None:
                for target in targets_elem.findall(f'{ns}target'):
                    polypeptide = target.find(f'{ns}polypeptide')
                    if polypeptide is not None:
                        # 策略1: 直接在 XML 中寻找 GenBank Gene Database (通常即 Entrez ID)
                        ext_ids = polypeptide.find(f'{ns}external-identifiers')
                        found_entrez = False
                        if ext_ids is not None:
                            for ext_id in ext_ids.findall(f'{ns}external-identifier'):
                                resource = ext_id.find(f'{ns}resource').text
                                if resource == 'GenBank Gene Database':
                                    identifier = ext_id.find(f'{ns}identifier').text
                                    entrez_targets.append(str(identifier).strip())
                                    found_entrez = True
                                    break # 找到 Entrez 就可以跳出当前靶点
            
            if entrez_targets:
                drug_to_entrez[db_id] = entrez_targets
                
            elem.clear() # 释放内存
            
    print(f"  ✅ 成功从 XML 直接提取到 {len(drug_to_entrez)} 种药物的 Entrez 靶点！")

    # 3. 连通性测试与注入
    edge_src, edge_dst = [], []
    for d_id, g_ids in drug_to_entrez.items():
        if d_id in drug_id_to_idx:
            u = drug_id_to_idx[d_id]
            for g_id in g_ids:
                if g_id in gene_id_to_idx: # 必须在图的基因节点空间内！
                    v = gene_id_to_idx[g_id]
                    edge_src.append(u)
                    edge_dst.append(v)
                    
    if len(edge_src) > 0:
        edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
        data['drug', 'targets', 'gene'].edge_index = edge_index
        torch.save(data, PT_PATH)
        print(f"🎉 完美注入！新增 ('drug', 'targets', 'gene') 边数: {edge_index.shape[1]}")
        print("💡 'Drug-Gene-Disease' 高阶视图正式打通，没有发生图断裂！")
    else:
        print("⚠️ 未能成功连边。可能是因为提取到的 Entrez ID 与图中的 CTD Gene ID 交集为 0。请检查 ID 格式。")

if __name__ == "__main__":
    extract_and_map_targets()

🧬 开始从 DrugBank 提取靶点并强制映射至 Entrez ID...
❌ 错误：图中未找到 node_id 属性，无法完成 ID 对齐！请检查图构建代码。


In [ ]:
import os
import torch
import xml.etree.ElementTree as ET
from tqdm import tqdm

# ================= 配置路径 =================
DATA_DIR = "/hy-tmp/else/data/data"
PT_PATH = "/hy-tmp/else/processed/final_hetero_data_strict.pt"
DRUGBANK_XML = os.path.join(DATA_DIR, "full database.xml")
# ============================================

def extract_and_map_targets():
    print("🧬 开始从 DrugBank 提取靶点并强制映射至 Entrez ID...")
    
    # 1. 载入当前的图数据
    data = torch.load(PT_PATH, weights_only=False)
    # 假设你的图中保留了原始 ID，比如 data['drug'].node_id 和 data['gene'].node_id
    # 如果没有，你需要根据原 CSV 文件列表生成字典
    if not hasattr(data['drug'], 'node_id') or not hasattr(data['gene'], 'node_id'):
        print("❌ 错误：图中未找到 node_id 属性，无法完成 ID 对齐！请检查图构建代码。")
        return
        
    drug_id_to_idx = {str(d): i for i, d in enumerate(data['drug'].node_id)}
    gene_id_to_idx = {str(g).replace('.0', '').strip(): i for i, g in enumerate(data['gene'].node_id)}
    
    print(f"  -> 图中识别到 {len(drug_id_to_idx)} 个药物，{len(gene_id_to_idx)} 个 Entrez 基因")

    ns = '{http://www.drugbank.ca}'
    drug_to_entrez = {}
    
    print("  -> 正在流式解析并映射 DrugBank XML (双重保险策略)...")
    context = ET.iterparse(DRUGBANK_XML, events=('end',))
    
    for event, elem in context:
        if elem.tag == f'{ns}drug' and (elem.getparent() is None or elem.getparent().tag == f'{ns}drugbank'):
            # 获取 DrugBank ID
            db_id_elem = elem.find(f'{ns}drugbank-id[@primary="true"]')
            if db_id_elem is None:
                elem.clear(); continue
            db_id = db_id_elem.text
            
            entrez_targets = []
            targets_elem = elem.find(f'{ns}targets')
            
            if targets_elem is not None:
                for target in targets_elem.findall(f'{ns}target'):
                    polypeptide = target.find(f'{ns}polypeptide')
                    if polypeptide is not None:
                        # 策略1: 直接在 XML 中寻找 GenBank Gene Database (通常即 Entrez ID)
                        ext_ids = polypeptide.find(f'{ns}external-identifiers')
                        found_entrez = False
                        if ext_ids is not None:
                            for ext_id in ext_ids.findall(f'{ns}external-identifier'):
                                resource = ext_id.find(f'{ns}resource').text
                                if resource == 'GenBank Gene Database':
                                    identifier = ext_id.find(f'{ns}identifier').text
                                    entrez_targets.append(str(identifier).strip())
                                    found_entrez = True
                                    break # 找到 Entrez 就可以跳出当前靶点
            
            if entrez_targets:
                drug_to_entrez[db_id] = entrez_targets
                
            elem.clear() # 释放内存
            
    print(f"  ✅ 成功从 XML 直接提取到 {len(drug_to_entrez)} 种药物的 Entrez 靶点！")

    # 3. 连通性测试与注入
    edge_src, edge_dst = [], []
    for d_id, g_ids in drug_to_entrez.items():
        if d_id in drug_id_to_idx:
            u = drug_id_to_idx[d_id]
            for g_id in g_ids:
                if g_id in gene_id_to_idx: # 必须在图的基因节点空间内！
                    v = gene_id_to_idx[g_id]
                    edge_src.append(u)
                    edge_dst.append(v)
                    
    if len(edge_src) > 0:
        edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
        data['drug', 'targets', 'gene'].edge_index = edge_index
        torch.save(data, PT_PATH)
        print(f"🎉 完美注入！新增 ('drug', 'targets', 'gene') 边数: {edge_index.shape[1]}")
        print("💡 'Drug-Gene-Disease' 高阶视图正式打通，没有发生图断裂！")
    else:
        print("⚠️ 未能成功连边。可能是因为提取到的 Entrez ID 与图中的 CTD Gene ID 交集为 0。请检查 ID 格式。")

if __name__ == "__main__":
    extract_and_map_targets()